# Notebook ULTRA-Détaillé - Gestionnaire de Budget : Finance Zen
# (Version Débutant Absolu - Chaque ligne est expliquée)

---

## 🎯 C'est quoi ce projet exactement ?

C'est une **application web complète** qui te permet de gérer ton argent. Concrètement :
- Tu peux **ajouter** des dépenses (courses, loyer...) et des revenus (salaire...)
- Tu vois ton **solde en temps réel** (Revenus - Dépenses)
- Tu as un **graphique en donut** (Chart.js) qui te montre visuellement où va ton argent
- Tu peux **filtrer** l'historique par catégorie
- Tu peux **switcher** entre le mode clair et le mode sombre

### 🧠 Pourquoi c'est un excellent exercice ?

Parce qu'on touche à TOUT ce qu'un développeur web junior doit maîtriser :

| Compétence | Ce qu'on apprend ici | Pourquoi c'est important en entreprise |
|---|---|---|
| **React & Vite** | Composants, hooks, état local | C'est LE framework Front-end le plus demandé aujourd'hui |
| **TypeScript** | Types, interfaces | Evite des bugs monstrueux en production |
| **Backend Node** | API REST avec Express.js | Savoir créer un serveur "Facteur" qui traite les requêtes |
| **Communication** | fetch, Appels HTTP, protocol API | Apprendre comment le front et le back se "parlent" ensemble via Internet |
| **Persistance** | Sauvegarder dans un fichier JSON | Comprendre les bases exactes du fonctionnement d'une Base de Données |
| **CSS UI/UX** | CSS Variables, thèmes dynamiques | Faire une interface "pro" et moderne sans s'arracher les cheveux |

> Si tu ne connais pas encore React ou TypeScript à 100%, PAS DE PANIQUE ! Ce guide détaille chaque concept comme si c'était la première fois.

---

# 🏗️ PARTIE 1 : L'Architecture Visuelle (Comment tout est relié)

## L'analogie du Restaurant (Modèle Client-Serveur)

Pour comprendre comment cette application web marche, on utilise toujours l'exemple du restaurant :
- **Le Client** (Toi à table) = Ton **navigateur web** (Chrome, Firefox). Il affiche l'interface React.
- **Le Serveur** (Le cuistot en cuisine) = Le **backend Express.js**.
- **Le Menu** que tu lis = Le **frontend React** qui te montre le formulaire et le Donut.
- **Le Serveur vocal** (Le serveur en salle) = **L'API** via `fetch()` qui transmet tes clics vers la cuisine.
- **Le Frigo** = Le **fichier `transactions.json`** qui garde les vraies données bien au frais sur le disque dur.

### 🔄 Le flux exact quand tu cliques sur "Ajouter"

```mermaid
graph TD;
    A[1. Tu remplis le formulaire React] -->|Clic Ajouter| B(2. React fabrique du JSON)
    B -->|fetch POST| C{3. Express.js reçoit}
    C -->|Il lit et insère en 1er| D[4. data/transactions.json]
    D -->|Confirmation du disque| C
    C -->|Retour HTTP 201 Created| B
    B -->|Mise à jour Reducer| E[5. React redessine le graphique et le Solde]
```

---

# 📦 PARTIE 2 : Les Fichiers de Configuration (Les fondations)

## 2.1 - `package.json` - La Carte d'Identité

C'est le fichier **le plus important** de Node.js. C'est le moteur du projet. Il liste tout ce dont on a besoin pour démarrer (comme la liste des courses).

```json
"scripts": {
  "dev": "vite",
  "server": "node server.js"
},
"dependencies": {
  "chart.js": "^4.4.7",     <-- Bibliotheque pour le dessin Graphique
  "cors": "^2.8.6",         <-- Bouclier de sécurité pour l'API
  "express": "^5.2.1",      <-- Notre createur de Serveur Backend NodeJS
  "react": "^19.2.0"        <-- Notre framework des ecrans/composants
}
```
On lance l'application via les `scripts` déclarés ici. Taper `npm run dev` ordonne à NPM d'activer l'outil "Vite".

### 2.2 - Le problème du proxy, expliqué simplement
React tourne sur ton PC sur la **porte (port) numéro 5173**. 
Express.js (le backend) tourne sur la **porte numéro 3001**.
Le navigateur web a une sécurité (CORS) qui **interdit formellement** à React d'envoyer un colis par dessus le mur chez le voisin du port 3001 (Piratage!).

La solution ? Le `vite.config.ts` !
```typescript
// fichier: vite.config.ts
server: {
    proxy: {
        '/api': 'http://localhost:3001'
    }
}
```
Ce code crée un pont souterrain (Le Proxy Vite). Quand React fait une demande commençant par `/api` (ex: `/api/transactions`), Vite l'attrappe, et l'envoie **en cachette** au backend NodeJS sur le 3001. Le navigateur n'a rien vu, c'est légal !


---

# 🍳 PARTIE 3 : `server.js` - Le Cerveau Backend et l'API

Ceci est notre cuisine. Express.js va nous permettre d'écouter les commandes des clients pour lire ou modifier notre frigo (`transactions.json`).

### L'Écriture Backend (ES Modules)
```javascript
import express from 'express'
import fs from 'fs'
import path from 'path'
// ...
```
On importe Express (Le serveur Web) et `fs` (File System, le lecteur de disque dur de NodeJS).

### Ouvrir le Frigo (Lecture du Fichier JSON)
```javascript
function readTransactions() {
    try {
        if (!fs.existsSync(DATA_FILE)) return [] // Si rien, on demarre a vide
        const data = fs.readFileSync(DATA_FILE, 'utf-8') // Lit le Texte (.json)
        return JSON.parse(data) // Convertit le Texte en un VRAI objet Javascript
    } catch (err) {
        return []
    }
}
```
Cette fonction est primordiale ! `readFileSync` stop le temps, et recupère toute les donnés. 
`JSON.parse` est fondamental car un fichier JSON est du pur texte, sans ça JavaScript ne le comprendrait pas comme un "Tableau" itérable.

### L'API (Comment on reçoit les commandes POST)
```javascript
app.post('/api/transactions', (req, res) => {
    // 1. Lire le frigo
    const transactions = readTransactions()
    
    // 2. Extraire la nouvelle commande du client (ce que t'a tapé)
    const newTransaction = req.body

    // 3. Fabriquer à la main ce qui lui manque : L'ID (code-bar)
    if (!newTransaction.id) {
        newTransaction.id = Date.now().toString() + Math.random().toString(36).substring(2)
    }
    // 4. Inserer ce nouvel achat AU DEBUT de la liste (le plus récent)
    transactions.unshift(newTransaction) 

    // 5. Fermer le frigo et le re-sauvegarder physiquement.
    writeTransactions(transactions)

    // 6. Crier "C'est prêt !" (201 Created)
    res.status(201).json(newTransaction)
})
```
Chaque appel React passera obligatoirement par cette route Express, modifiant le fichier texte en temps réel, garantissant que tes données survivront à un redémarrage d'ordinateur ou fermeture d'onglet.

---

# ⚛️ PARTIE 4 : React - Le Super Chef d'Orchestre ! (`App.tsx`)

C'est ici que React bat son plein. Dans `App.tsx`, on rassemble l'historique, le formulaire, et on calcule le solde final pour l'afficher à l'écran !

### C'est quoi un Hook `useReducer` ?
Un composant React doit "mémoriser" des données pour fonctionner. S'il utilise `useState`, chaque fonction fera la même chose bêtement (Ajouter.. Mettre à jour.. Detruire...). Quand on a un tableau de 50 transactions, React conseille plutôt d'utiliser son grand-frère : le `useReducer` (L'Architecte des données).

```typescript
// Le switch case gère l'état entier de tes portefeuilles comme une Banque.
function budgetReducer(state: BudgetState, action: BudgetAction): BudgetState {
    switch (action.type) {
        case 'ADD_TRANSACTION': {
            // On "DESTRUCTURE" l'ancien tableau pour le coller avec le nouveau.
            const updated = [action.payload, ...state.transactions]
            return { transactions: updated }
        }
        case 'DELETE_TRANSACTION': {
            // Si id = 8, alors je filtre l'ancien tableau en n'incluant PLUS le numéro 8.
            const updated = state.transactions.filter(t => t.id !== action.payload)
            return { transactions: updated }
        }
    }
}
```
**La Règle d'Or en React : L'Immutabilité.**
Tu n'as jamais le droit de modifier brutalement une donnée : `state.transactions.push(nouveau)`. C'est formellement interdit ! React vérifiera **les références mémoires**. Si tu pousses brutalement, React jugera que la mémoire n'a pas bougé de place, et ne ré-affichera **JAMAIS** le graphique !
Pour qu'il redessine l'écran, il faut un tableau frais et neuf : `[Nouveau, ...L'AncienTableauDécoupé]`.

### useEffect() - La synchronisation avec le monde exterieur
Quand le composant React démarre, la boite aux transactions est vide (`{transactions: []}`). Comment la remplir via notre Backend NodeJS sans que tout explose ?
```typescript
    useEffect(() => {
        // Ceci va recracher une Promise (du javascript Asynchrone !)
        loadTransactions().then(loaded => {
            dispatch({ type: 'LOAD_TRANSACTIONS', payload: loaded })
        })
    }, [])  // <-- IMPORTANT : LE TABLEAU VIDE
```
Le tableau vide `[]` à la fin d'`useEffect` est vital. Ça murmure à React : "*Fais le HTTP `fetch` pour récupérer le JSON au démarrage, **UNE SEULE FOIS DANS TA VIE**, puis arrête-toi.*
S'il n'y avait pas ce tableau, `useEffect` ferait l'appel HTTP 500 fois par seconde, explosant ton PC par une boucle infinie de rendus.

### Exécuteur d'Event : Ajouter le Formulaire (Lier Front et Back)
```typescript
    async function handleAddTransaction(name, amount, type, category) {
        const newTransaction = { id: "-", name, amount, type... } // Construit la structure
        try {
            // 1. Envoyer à l'API ! (Le code patiente le "await")
            await fetch('/api/transactions', { 
                method: 'POST',
                headers: { 'Content-Type': 'application/json' }, // Je parle le dialecte JSON
                body: JSON.stringify(newTransaction),
            })
            // 2. Si le serveur NodeJS (Port 3001) dit OUI, on informe React de se mettre a jour
            dispatch({ type: 'ADD_TRANSACTION', payload: newTransaction })
        } catch { 
            console.error("Le serveur backend NodeJS n'est surement pas allumé.")
        }
    }
```

---

# 🖌️ PARTIE 5 : Composants de Détails (`BudgetChart.tsx`)

Dans `BudgetChart`, on intégre **Chart.js**, une librairie monstrueuse non-pensée pour React.

### Rendu du Canvas et Danger
React manipule le DOM HTML. Mais Chart.js aime **AUSSI** manipuler les canvas sauvagement. Si les deux s'affrontent, la page clignottera de bogues visuels atroces.

**L'astuce de l'Artisan avec `useRef`**
```typescript
    // Corde physique rattaché a l'objet html du canevas
    const chartRef = useRef<HTMLCanvasElement>(null)
    // Boite noire en mémoique qui echape au recompte React pour garder l'instance Chart alive
    const chartInstance = useRef<Chart | null>(null)
    
    useEffect(() => {
        // Si un vieux Graphique existait... CRASH LE SAUVAGEMENT d'abord.
        if (chartInstance.current) {
            chartInstance.current.destroy()
        }
        // ... Regroupements des additions ...
        
        // Enfin récréer le saint Graal d'un coup.
        chartInstance.current = new Chart(chartRef.current, { type: 'doughnut', ...data})
        
        // La fonction clean-up (Si le composant disparait, on detruit pour eviter la fuite de mémoire)
        return () => { if (chartInstance.current) chartInstance.current.destroy() }
    }, [chartTransactions, theme])
```
Ce bloc est un chef d'oeuvre de maitrise React : On écoute les transactions (Si j'achète une PS5, `[chartTransactions]` se réveille), on DÉTRUIT violement l'ancien diagramme `ChartJS`, et on en relance un flambant neuf en 1 ms. Ni vu ni connu !

---

# 💅 PARTIE 6 : Finition Magique au CSS (`index.css`)

Cette application intègre le concept du **Glassmorphisme** ainsi que du changement dynamique global de Thème, géré à un seul en-droit grace au `:root` CSS !

### C'est quoi un CSS variable ?
```css
:root {
    /* Thème Bois (La norme) */
    --bg-primary: #f5efe6;
    --accent: #8b6f47;
}

/* Si par chance l'element <html> (Le boss final au dessus de tout) possède data-theme="dark", 
   alors LES MEMES VARIABLES SE FONT ÉCRASER de force partout ! */
[data-theme="dark"] {
    --bg-primary: #1a1410;
    --accent: #c9a96e;
}
```
Cette technique est fantastique : Pas besoin de coder 300 lignes pour faire des classes en ".body-dark-mode" ou ".header-sombre-fond". Tout est mutualisé ! Dans tes fenêtres (les `.panel`), l'arrière plan appelera TOUJOURS `background: var(--bg-card)`. Que ce soit noir pur ou blanc pur selon l'HTML en haut, le panel obéira au doigt et a l'oeil.

### Transparence de Verre (Glassmorphisme)
```css
.panel {
    background: rgba(210, 190, 165, 0.35);   /* L'opacité 4ème paramètre à 0.35 permet la semi-transparence */
    backdrop-filter: blur(12px);               /* Magie qui emmêle et FLoute pixels physiquement cachés en dessous */
    border-radius: 14px;                       
}
```
Et voici ! Ton Dashboard de Budget est créé de A à Z par ta maîtrise complète du Front, de l'API Node, du Back JSON et du CSS Dynamique !